In [0]:
from pyspark.sql import functions as F

In [0]:
%sql
use catalog test_catlog

In [0]:
data = [
    (1, 2, 3),
    (4, 5, 6),
    (7, 8, 9)
]

df = spark.createDataFrame(data, ["a", "b", "c"])
df.write.saveAsTable("test_schema.test_tbl", mode="overwrite")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS test_catlog.test_schema;
CREATE VOLUME IF NOT EXISTS test_catlog.test_schema.test_volume1;

In [0]:
df.write.mode("append").format("parquet").save("/Volumes/test_catlog/test_schema/test_volume")

In [0]:
display(
    spark.read.format("parquet").load("/Volumes/test_catlog/test_schema/test_volume")
)

In [0]:
data = [
    (1, 2, 3, 4)
]

df = spark.createDataFrame(data, ["a", "b", "c", "d"])

display(df)

In [0]:
dbutils.fs.ls("/Volumes/test_catlog/test_schema/test_volume1")

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS test_schema.processed

In [0]:
dbutils.fs.mv("/Volumes/test_catlog/test_schema/test_volume1", "/Volumes/test_catlog/test_schema/processed", True)

In [0]:
dbutils.fs.cp("/Volumes/test_catlog/test_schema/processed", "/Volumes/test_catlog/test_schema/test_volume", True)

In [0]:
dbutils.fs.rm("/Volumes/test_catlog/test_schema/test_volume", True)

In [0]:
dbutils.fs.mkdirs("/Volumes/test_catlog/test_schema/test_volume/12345")

In [0]:
dbutils.fs.cp("/Volumes/test_catlog/test_schema/processed/part-00000-tid-3678108402199454968-4cf0642c-9f64-43c6-a40c-2daa1fc2f062-221-1.c000.snappy.parquet","/Volumes/test_catlog/test_schema/test_volume/12345/test.snappy.parquet")

In [0]:
from multiprocessing.pool import ThreadPool

source_dir = "/Volumes/test_catlog/test_schema/test_volume"
target_dir = "/Volumes/test_catlog/test_schema/test_volume1/12345"

# Get the list of file paths
files_to_move = [f.path for f in dbutils.fs.ls(source_dir) if f.name.endswith(".parquet")]

def move_file(path):
    file_name = path.split('/')[-1]
    dbutils.fs.mv(path, target_dir + file_name, True)


In [0]:
[f.path for f in dbutils.fs.ls(source_dir) if f.name.endswith(".parquet")]

In [0]:
# Use a ThreadPool to run moves in parallel (e.g., 16 at a time)
pool = ThreadPool(500)
pool.map(move_file, files_to_move)
pool.close()
pool.join()

In [0]:
files = [f.path for f in dbutils.fs.ls(source_dir) if f.name.endswith(".parquet")]
df = spark.createDataFrame(files, "string").toDF("path")

df.repartition(100)

display(df)

In [0]:
def move_partition(iterator):
    for path in iterator:
        try:
            file_name = path.split("/")[-1]
            dbutils.fs.mv(path, target_dir + file_name, True)
        except Exception as e:
            print(f"Failed for {path}: {e}")
df.foreachPartition(move_partition)            

In [0]:
df.foreachPartition(move_partition)